<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">وزن ثابت، انتخاب متفاوت</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چرا یک <bdi dir="ltr">Prompt</bdi> با چند قانون <bdi dir="ltr">Sampling</bdi> ادامه‌های متفاوت می‌گیرد؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-09/chapter-01/54-generate.html"><bdi dir="ltr">54-generate</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/55-temperature.html"><bdi dir="ltr">55-temperature</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-09/chapter-02/56-topkp.html"><bdi dir="ltr">56-topkp</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-09/chapter-03/57-prompts.html"><bdi dir="ltr">57-prompts</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این دفتر مستقل است و مدل کوچک خودش را در ۸۰ گام آموزش می‌دهد؛ به اجرای دفتر آموزش یا <bdi dir="ltr">Checkpoint</bdi> آن نیاز ندارد. پس از این مرحله وزن‌ها ثابت می‌مانند. هدف مقایسهٔ قانون انتخاب است، نه رسیدن به متن باکیفیت یا دستیار قابل اعتماد.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.data import prepare_corpus
from mini_gpt.dataset import NextTokenDataset
from mini_gpt.model import MiniGPT
from mini_gpt.train import random_batch
train_ids,valid_ids,tokenizer,metadata = prepare_corpus(ROOT/"data"/"sample.txt",train_fraction=0.8)
config = ModelConfig(vocab_size=tokenizer.vocab_size,context_length=16,
                     embedding_dim=32,num_heads=4,num_layers=1,dropout=0.1)
model = MiniGPT(config).cpu()
optimizer = torch.optim.AdamW(model.parameters(),lr=0.003,weight_decay=0.01)
train_data = NextTokenDataset(train_ids,config.context_length)
valid_data = NextTokenDataset(valid_ids,config.context_length)
batch_rng = torch.Generator().manual_seed(18)
prompt_text = "مدل "
prompt = torch.tensor([tokenizer.encode(prompt_text)],dtype=torch.long)
assert 0 not in prompt[0].tolist()
print("Vocabulary:",list(enumerate(tokenizer.id_to_token)))
print("Parameters:",sum(p.numel() for p in model.parameters()))
print("Train/validation windows:",len(train_data),len(valid_data))
print("Validation unknown rate:",metadata["validation_unknown_rate"])


In [ ]:
for step in range(1,81):
    model.train()
    x,y = random_batch(train_data,batch_size=8,generator=batch_rng)
    optimizer.zero_grad(set_to_none=True)
    _,loss = model(x,y)
    if not torch.isfinite(loss):
        raise RuntimeError("Nonfinite Loss")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True)
    optimizer.step()
model.eval()
print("Training finished; last batch Loss:",loss.item())


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">گام اول را ثابت نگه داریم</h2><p style="text-align:right">ابتدا یک ورودی و یک مجموعه <bdi dir="ltr">Logits</bdi> ثابت را با چند قانون انتخاب می‌سنجیم. پیش‌بینی کنید <bdi dir="ltr">Greedy</bdi>، <bdi dir="ltr">Temperature</bdi> بالاتر، <bdi dir="ltr">Top-k</bdi> و <bdi dir="ltr">Top-p</bdi> چگونه مجموعهٔ نامزدها و احتمال‌ها را تغییر می‌دهند. <bdi dir="ltr">Greedy</bdi> را با <bdi dir="ltr">Temperature</bdi>=0 پیاده نمی‌کنیم. نمودار فقط هشت <bdi dir="ltr">Token</bdi> با بیشترین امتیاز اولیه را نشان می‌دهد؛ مجموع احتمال‌های نمایش‌داده‌شده ممکن است کمتر از یک باشد.</p>
</div>

In [ ]:
from mini_gpt.sampling import sampling_distribution
methods = {
    "greedy": {"greedy":True},
    "temperature=0.7": {"temperature":0.7},
    "temperature=1.3": {"temperature":1.3},
    "top-k=8": {"top_k":8},
    "top-p=0.8": {"top_p":0.8},
}
with torch.no_grad():
    logits = model(prompt[:,-config.context_length:])[0][:,-1,:]
inspect("last-position logits",logits)
shown_ids = logits[0].argsort(descending=True)[:8].tolist()
print("Shown IDs:",shown_ids)
print("Tokens:",[repr(tokenizer.id_to_token[i]) for i in shown_ids])
fig,ax = plt.subplots(figsize=(8,4))
for name,options in methods.items():
    probabilities = sampling_distribution(logits,**options)[0]
    torch.testing.assert_close(probabilities.sum(),torch.tensor(1.))
    print(name,"candidate count:",torch.count_nonzero(probabilities).item())
    ax.plot(range(len(shown_ids)),[probabilities[i].item() for i in shown_ids],"o-",label=name)
ax.set_xticks(range(len(shown_ids)),[str(i) for i in shown_ids])
ax.set(xlabel="Token ID, sorted by raw score",ylabel="Sampling probability")
ax.legend()
plt.show()


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">حالا ادامهٔ واقعی تولید کنیم</h2><p style="text-align:right">از <bdi dir="ltr">generate</bdi> واقعی استفاده می‌کنیم. دو <bdi dir="ltr">Seed</bdi> برای روش‌های تصادفی می‌گذاریم. یکسان‌بودن <bdi dir="ltr">Seed</bdi> میان روش‌ها تضمین خروجی یکسان نیست؛ توزیع عوض شده است. بعد از نخستین اختلاف، خود زمینهٔ ادامه هم متفاوت می‌شود.</p>
</div>

In [ ]:
before = {name:value.detach().clone() for name,value in model.state_dict().items()}
for name,options in methods.items():
    for seed in ((23,) if options.get("greedy") else (23,29)):
        torch.manual_seed(seed)
        generated = model.generate(prompt,max_new_tokens=48,**options)
        inspect(name,generated)
        print(name,"seed",seed)
        print(tokenizer.decode(generated[0].tolist()))
assert all(torch.equal(before[name],value) for name,value in model.state_dict().items())
try:
    model.generate(prompt,max_new_tokens=1,temperature=0)
except ValueError as error:
    print("Expected invalid temperature:",error)
else:
    raise AssertionError("Temperature must be positive")


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">زمینهٔ بلند کجا بریده می‌شود؟</h2><p style="text-align:right"><bdi dir="ltr">generate</bdi> زمینهٔ ورودی <bdi dir="ltr">Forward</bdi> را به طول مجاز می‌برد، اما متن برگشتی شامل تمام <bdi dir="ltr">Prompt</bdi> و ادامه است. در این پروژه، شمارهٔ موقعیت‌های پنجرهٔ بریده‌شده دوباره از صفر شروع می‌شود؛ جدول موقعیت نیز با همین شماره‌ها خوانده می‌شود.</p>
</div>

In [ ]:
long_prompt = prompt.repeat(1,6)
with torch.no_grad():
    next_logits = model(long_prompt[:,-config.context_length:])[0][:,-1,:]
    expected_id = next_logits.argmax(-1).item()
    generated = model.generate(long_prompt,max_new_tokens=1,greedy=True)
assert generated.shape[1] == long_prompt.shape[1]+1
assert generated[0,-1].item() == expected_id
print("Full prompt length:",long_prompt.shape[1],"model context:",config.context_length)


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین و برداشت:</b> احتمال گام اول، یک ادامهٔ <bdi dir="ltr">Greedy</bdi> و دو ادامه برای هر روش تصادفی را مقایسه کنید. کدام تفاوت را می‌توان به <bdi dir="ltr">Sampling</bdi> نسبت داد؟ برای داوری دربارهٔ کیفیت، چه نمونه‌ها و معیارهای بیشتری لازم است؟ <bdi dir="ltr">Temperature</bdi> وزن‌ها را آموزش نمی‌دهد و روان‌بودن یا تنوع، درستی پاسخ را تضمین نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-09/chapter-03/57-prompts.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: احتمال با فراوانی یک اجرای کوتاه یکی نیست</h2>
<p style="text-align:right">از توزیع واقعی <bdi dir="ltr">Sampling</bdi> نمونه بگیرید و فراوانی‌ها را بدون ادعای برابری دقیق بررسی کنید. پیش‌نیاز: <bdi dir="ltr">Temperature</bdi>، <bdi dir="ltr">Top-k</bdi> و توزیع گام اول در همین دفتر را بشناسید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر یک <bdi dir="ltr">Token</bdi> احتمال صفر داشته باشد، آیا باید در شمارش دیده شود؟ آیا <bdi dir="ltr">Token</bdi> با احتمال ۰٫۲ در ۱۰ نمونه حتماً دو بار ظاهر می‌شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.sampling import sampling_distribution
review_logits = torch.tensor([[2.,1.,0.,-1.]])
review_probabilities = sampling_distribution(review_logits,top_k=3)[0]
print('fixed one-step distribution:',review_probabilities)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">sample_counts(probabilities,draws,generator)</code> با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">torch.multinomial</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">replacement=True</code> نمونه بگیرد و یک <bdi dir="ltr">Tensor</bdi> شمارش طول <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">V</code> برگرداند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">probabilities</code> یک بردار است؛ این‌ها تکرارِ مستقلِ همان گام ثابت‌اند، نه تولید یک دنبالهٔ جدید.</p>
</div>

In [ ]:
def sample_counts(probabilities, draws, generator):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = sample_counts(review_probabilities,100,torch.Generator().manual_seed(23))
    if result is None: return False
    assert result.shape == review_probabilities.shape and result.sum().item() == 100
    assert result[-1].item() == 0
    expected_ids = torch.multinomial(review_probabilities,100,replacement=True,generator=torch.Generator().manual_seed(23))
    assert torch.equal(result,torch.bincount(expected_ids,minlength=4))
    assert sample_counts(torch.tensor([0.,1.,0.]),7,torch.Generator().manual_seed(2)).tolist() == [0,7,0]
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط تعداد نمونه‌ها را از ۲۰ به ۲۰۰ و ۲۰۰۰ تغییر دهید. اختلاف تجربی را گزارش کنید؛ نزدیک‌شدن معمول را با کاهش قطعی و یکنواخت خطا در هر اجرای تصادفی اشتباه نگیرید.</p>
</div>

In [ ]:
for draws in (20,200,2000):
    sampled = torch.multinomial(review_probabilities,draws,replacement=True,generator=torch.Generator().manual_seed(31))
    frequency = torch.bincount(sampled,minlength=4).float()/draws
    print('draws, frequency, maximum error:',draws,frequency,(frequency-review_probabilities).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">کد خراب <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">argmax</code> را بارها تکرار می‌کند و آن را <bdi dir="ltr">Sampling</bdi> می‌نامد. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">draw_ids(probabilities,draws,generator)</code> شناسه‌های نمونه‌برداری‌شده را برگرداند؛ از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">generator</code> داده‌شده استفاده کنید.</p>
</div>

In [ ]:
wrong = review_probabilities.argmax().repeat(20)
print('greedy is not repeated random sampling:',torch.bincount(wrong,minlength=4))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def draw_ids(probabilities, draws, generator):
    # TODO
    return None

In [ ]:
def test_repair():
    result = draw_ids(review_probabilities,40,torch.Generator().manual_seed(7))
    if result is None: return False
    expected = torch.multinomial(review_probabilities,40,replacement=True,generator=torch.Generator().manual_seed(7))
    assert torch.equal(result,expected)
    assert draw_ids(torch.tensor([0.,0.,1.]),3,torch.Generator().manual_seed(1)).tolist() == [2,2,2]
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">توزیع از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">sampling_distribution</code> واقعی پروژه گرفته شد. در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">generate</code> پس از هر انتخاب، <bdi dir="ltr">Context</bdi> و <bdi dir="ltr">Logits</bdi> دوباره تغییر می‌کنند؛ پس فراوانی <bdi dir="ltr">Token</bdi>های یک متن بلند، آزمایش همین توزیع ثابت نیست.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">برای مقایسهٔ منصفانهٔ قانون انتخاب، چرا دفتر ابتدا یک مجموعه <bdi dir="ltr">Logits</bdi> ثابت را بررسی می‌کند و بعد سراغ ادامهٔ متن می‌رود؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-03/57-prompts.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-12_sampling.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>